# Data Preprocessing Pipeline: Inflation Rate by Country

This notebook transforms raw JSON data into a structured and normalized format.

**Input:** Raw JSON dataset  
**Processing:** Type casting, column standardization, missing value handling, validation  
**Output:** Cleaned JSON dataset prepared for database upload

## Data description
Inflation as measured by the consumer price index reflects the annual percentage change in the cost to the average consumer of acquiring a basket of goods and services that may be fixed or changed at specified intervals, such as yearly. The Laspeyres formula is generally used.

### Fields:
- __country_name__
- __country_code__ - country name in standards ISO 3
- __years 2023-2019__ - inflation rate in the appropriate year

In [69]:
# Importing libraries
import pandas as pd
import pycountry
import matplotlib.pyplot as plt

In [71]:
# Helper function to transform country names to ISO3 (standard for all datasets in our project)
def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

In [73]:
df = pd.read_json('raw_data\inflation_raw.json') # Check the directory where raw data file is stored

<>:1: SyntaxWarning: invalid escape sequence '\i'
<>:1: SyntaxWarning: invalid escape sequence '\i'
C:\Users\Honor\AppData\Local\Temp\ipykernel_17396\2798543537.py:1: SyntaxWarning: invalid escape sequence '\i'
  df = pd.read_json('raw_data\inflation_raw.json') # Check the directory where raw data file is stored


In [75]:
# glancing at the data
df.head()

,Country Name,2023,2022,2021,2020,2019
0,Lebanon,221.34%,171.21%,154.76%,84.86%,3.01%
1,Turkey,53.86%,72.31%,19.60%,12.28%,15.18%
2,Sierra Leone,47.64%,27.21%,11.87%,13.45%,14.81%
3,Iran,44.58%,43.49%,43.39%,30.59%,39.91%
4,Ghana,38.11%,31.26%,9.97%,9.89%,7.14%


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Country Name  177 non-null    object
 1   2023          177 non-null    object
 2   2022          177 non-null    object
 3   2021          177 non-null    object
 4   2020          177 non-null    object
 5   2019          177 non-null    object
dtypes: object(6)
memory usage: 8.4+ KB


In [79]:
df.describe()

,Country Name,2023,2022,2021,2020,2019
count,177,177,177,177,177,177
unique,177,147,160,157,157,156
top,Lebanon,0.00%,0.00%,0.00%,0.00%,1.81%
freq,1,24,9,7,7,3


In [81]:
# changing country name type to string and renaming a column to snake_case stype
df['country_name'] = df['Country Name'].astype('string')
df = df.drop(columns=["Country Name"])

In [85]:
# choosing only columns with years (all except country_name)
year_cols = df.columns.drop("country_name")

# removing % and casting to float
df[year_cols] = (
    df[year_cols]
        .replace("%", "", regex=True)
        .astype(float)
)

In [87]:
df.head()

,2023,2022,2021,2020,2019,country_name
0,221.34,171.21,154.76,84.86,3.01,Lebanon
1,53.86,72.31,19.60,12.28,15.18,Turkey
2,47.64,27.21,11.87,13.45,14.81,Sierra Leone
3,44.58,43.49,43.39,30.59,39.91,Iran
4,38.11,31.26,9.97,9.89,7.14,Ghana


In [99]:
df.describe()

,2023,2022,2021,2020,2019
count,177.000000,177.000000,177.000000,177.000000,177.000000
mean,8.487910,11.899831,8.232373,7.715989,5.265085
std,18.426646,19.420032,30.377223,44.016435,20.732476
min,-1.040000,-6.690000,-0.770000,-2.600000,-3.230000
25%,2.380000,5.090000,1.830000,0.220000,0.850000
50%,4.940000,7.830000,3.300000,1.780000,2.210000
75%,8.740000,11.580000,5.710000,3.780000,3.720000
max,221.340000,171.210000,359.090000,557.200000,255.310000


In [105]:
# transforming dataset in long format (better for uploading in DB)
df_long = pd.melt(df, id_vars = ['country_name'], value_vars = ['2019', '2020', '2021', '2022', '2023'])

In [103]:
df_long.head()

,country_name,variable,value
0,Lebanon,2019,3.01
1,Turkey,2019,15.18
2,Sierra Leone,2019,14.81
3,Iran,2019,39.91
4,Ghana,2019,7.14


In [107]:
# adding new column with standartized country name for joining with other datasets
df["country_code"] = df["country_name"].apply(to_iso3)
df_long["country_code"] = df_long["country_name"].apply(to_iso3)

In [109]:
df[df["country_code"].isna()]

,2023,2022,2021,2020,2019,country_name,country_code
1,53.86,72.31,19.60,12.28,15.18,Turkey,None
7,31.23,22.96,3.76,5.10,3.32,Lao PDR,None
74,5.87,3.74,1.24,-0.74,1.58,West Bank And Gaza,None
88,4.94,11.58,3.35,0.20,2.68,Kosovo,None
95,4.56,5.66,1.57,-0.63,0.91,St. Vincent And The Grenadines,None
98,4.30,3.04,1.72,1.80,2.21,Republic Of Congo,None
102,4.07,6.38,2.41,-1.76,0.54,St. Lucia,None
114,3.56,2.67,1.20,-1.17,-0.33,St. Kitts And Nevis,None
149,0.36,3.68,1.73,1.94,-0.39,Brunei,None
156,0.00,4.83,-0.01,-2.08,-1.93,UAE,None


In [111]:
df_long[df_long["country_code"].isna()]

,country_name,variable,value,country_code
1,Turkey,2019,15.18,None
7,Lao PDR,2019,3.32,None
74,West Bank And Gaza,2019,1.58,None
88,Kosovo,2019,2.68,None
95,St. Vincent And The Grenadines,2019,0.91,None
...,...,...,...,...
857,Brunei,2023,0.36,None
864,UAE,2023,0.00,None
866,Curacao,2023,0.00,None
868,Micronesia,2023,0.00,None


In [113]:
# Since not all country_names were mapped, I manually change them
manual_map = {
    "Turkey": "TUR",
    "Lao PDR": "LAO",
    "West Bank And Gaza": "PSE",
    "Kosovo": "XKX",
    "St. Vincent And The Grenadines": "VCT",
    "Republic Of Congo": "COG",
    "St. Lucia": "LCA",
    "St. Kitts And Nevis": "KNA",
    "Brunei": "BRN",
    "UAE": "ARE",
    "Curacao": "CUW",
    "Micronesia": "FSM",
    "Russia": "RUS"
}

df["country_code"] = df["country_code"].fillna(df["country_name"].map(manual_map))
df_long["country_code"] = df_long["country_code"].fillna(df_long["country_name"].map(manual_map))

In [115]:
df.head()

,2023,2022,2021,2020,2019,country_name,country_code
0,221.34,171.21,154.76,84.86,3.01,Lebanon,LBN
1,53.86,72.31,19.60,12.28,15.18,Turkey,TUR
2,47.64,27.21,11.87,13.45,14.81,Sierra Leone,SLE
3,44.58,43.49,43.39,30.59,39.91,Iran,IRN
4,38.11,31.26,9.97,9.89,7.14,Ghana,GHA


In [117]:
df_long.head()

,country_name,variable,value,country_code
0,Lebanon,2019,3.01,LBN
1,Turkey,2019,15.18,TUR
2,Sierra Leone,2019,14.81,SLE
3,Iran,2019,39.91,IRN
4,Ghana,2019,7.14,GHA


In [126]:
# renaming columns in the long table
df_long.rename(columns={
    "variable": "year",
    "value": "inflation_rate" }, inplace = True)

## adding a common column for manipulations with other tables (table merges): country_name + year
in the current dataset we take fields 'country_code' and 'year' to compose it

In [128]:
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885 entries, 0 to 884
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_name    885 non-null    string 
 1   year            885 non-null    object 
 2   inflation_rate  885 non-null    float64
 3   country_code    885 non-null    object 
dtypes: float64(1), object(2), string(1)
memory usage: 27.8+ KB


In [136]:
df_long['year'] = df_long['year'].astype('int')

In [138]:
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885 entries, 0 to 884
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_name    885 non-null    string 
 1   year            885 non-null    int32  
 2   inflation_rate  885 non-null    float64
 3   country_code    885 non-null    object 
dtypes: float64(1), int32(1), object(1), string(1)
memory usage: 24.3+ KB


In [140]:
df_long["country_year"] = (df_long["country_code"] + df_long["year"].astype(str))

In [142]:
df_long.head()

,country_name,year,inflation_rate,country_code,country_year
0,Lebanon,2019,3.01,LBN,LBN2019
1,Turkey,2019,15.18,TUR,TUR2019
2,Sierra Leone,2019,14.81,SLE,SLE2019
3,Iran,2019,39.91,IRN,IRN2019
4,Ghana,2019,7.14,GHA,GHA2019


## Saving preprocessed datasets

In [146]:
    df.to_json(
        "processed_data/inflation_rate_wide_clean.json",
        orient="records",
        indent=2,
        force_ascii=False
    )

In [148]:
df_long.to_json(
    "processed_data/inflation_rate_clean.json",
    orient="records",
    indent=2,
    force_ascii=False
)